### SerumGate H3N2 Strain Site Attention Visualization

This section visualizes SerumGate low-rank pooling attention by aligned HA residue site for the H3N2 strain subtype model. ESM embeddings contain CLS/EOS special tokens, so the extraction below removes those special-token weights before mapping token attention back to aligned HA sites. The site-level mean and max profiles are computed by re-running the trained model over the train+valid training pool and accumulating per-residue attention online.


In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

ROOT = Path("/home/chenyh/workspace/fluProfiler")
sys.path.append(str(ROOT / "src"))
sys.path.append(str(ROOT / "experiments" / "serum_gate"))

from fluprofiler.models.serum_gate_model import SerumGateModel
from train_zero_shot import (
    SerumGateVocabs,
    load_fixed_split_frames,
    required_embedding_files,
    validate_embedding_files,
    build_loader,
    move_batch,
)

RESULT_DIR = ROOT / "results/H1H3_new/SerumGate/strain/subtype/H3N2/refit_train_valid_e132_q32_s42_subtype0"
CKPT_PATH = RESULT_DIR / "checkpoints/best_model.pth"
OUT_DIR = RESULT_DIR / "attention_train_sites"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESIDUE_ATTENTION_PATH = OUT_DIR / "train_site_residue_attention_summary.csv"
SPECIAL_ATTENTION_PATH = OUT_DIR / "train_special_token_attention_summary.csv"

# None = full train+valid pool. Use a small integer only for debugging.
MAX_ROWS_FOR_DEBUG = None
FORCE_RECOMPUTE = False
PRINT_EVERY = 10
EXPECTED_ALIGNED_HA_LEN = 566
EXPECTED_MIN_RESIDUE_SITES = 500

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[info] device = {device}")
print(f"[info] result dir = {RESULT_DIR}")


In [ ]:
def is_valid_residue_attention_table(path: Path) -> tuple[bool, str]:
    if not path.is_file():
        return False, "file missing"
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        return False, f"cannot read csv: {exc}"
    required = {"role", "site", "mean_attention", "max_attention", "n"}
    if not required.issubset(df.columns):
        return False, f"missing columns: {sorted(required - set(df.columns))}"
    if set(df["role"].astype(str).unique()) != {"serum", "virus"}:
        return False, f"unexpected roles: {sorted(df['role'].astype(str).unique())[:10]}"
    numeric_site = pd.to_numeric(df["site"], errors="coerce")
    numeric_mean = pd.to_numeric(df["mean_attention"], errors="coerce")
    numeric_max = pd.to_numeric(df["max_attention"], errors="coerce")
    numeric_n = pd.to_numeric(df["n"], errors="coerce")
    if numeric_site.isna().any() or numeric_mean.isna().any() or numeric_max.isna().any() or numeric_n.isna().any():
        return False, "site/attention/n contains non-numeric values"
    if numeric_site.min() < 1 or numeric_site.max() > EXPECTED_ALIGNED_HA_LEN:
        return False, f"site range outside aligned HA residues: {int(numeric_site.min())}-{int(numeric_site.max())}"
    site_counts = df.assign(site=numeric_site.astype(int)).groupby("role")["site"].nunique()
    if (site_counts < EXPECTED_MIN_RESIDUE_SITES).any():
        return False, f"too few residue sites per role: {site_counts.to_dict()}"
    if not ((numeric_mean >= 0).all() and (numeric_mean <= 1).all() and (numeric_max >= 0).all() and (numeric_max <= 1).all()):
        return False, "attention values outside [0, 1]"
    if not (numeric_n > 0).all():
        return False, "some sites have non-positive counts"
    return True, "ok"


def lowrank_attention_weights(model, matrix, mask):
    pooler = model.ha_encoder.lowrank_pooler
    if pooler is None:
        raise RuntimeError("This model is not using lowrank_attention.")

    context = pooler.input_projection(pooler.norm(matrix))
    key_padding_mask = mask <= 0 if mask is not None else None
    context, _ = pooler.context_attention(
        context,
        context,
        context,
        key_padding_mask=key_padding_mask,
        need_weights=False,
    )
    logits = pooler.score(torch.tanh(pooler.dropout(context))).squeeze(-1)
    if mask is not None:
        logits = logits.masked_fill(mask <= 0, torch.finfo(logits.dtype).min)
    return torch.softmax(logits, dim=1)


def split_attention_to_aligned_residue_sites(aligned_ha, token_weights):
    """
    Map token attention to aligned HA residue sites.

    The stored ESM matrices are usually len(ungapped_HA) + 2 because they include
    CLS and EOS tokens. Those special tokens are reported separately and are not
    assigned to residue sites.
    """
    token_weights = np.asarray(token_weights, dtype=float)
    aligned_ha = str(aligned_ha)
    residue_sites = []
    residue_aas = []
    for aligned_idx, aa in enumerate(aligned_ha, start=1):
        if aa == "-":
            continue
        residue_sites.append(aligned_idx)
        residue_aas.append(aa)

    n_res = len(residue_sites)
    n_tokens = len(token_weights)
    special = {}
    if n_tokens == n_res + 2:
        residue_weights = token_weights[1:-1]
        special = {"cls": float(token_weights[0]), "eos": float(token_weights[-1])}
    elif n_tokens == n_res:
        residue_weights = token_weights
    else:
        raise ValueError(
            f"Cannot map token attention to HA sites: token_count={n_tokens}, "
            f"ungapped_residue_count={n_res}, aligned_len={len(aligned_ha)}"
        )

    if len(residue_weights) != n_res:
        raise RuntimeError("Internal mapping error: residue_weights and residue_sites length mismatch")
    return (
        np.asarray(residue_sites, dtype=int),
        np.asarray(residue_aas),
        np.asarray(residue_weights, dtype=float),
        special,
    )


def update_site_stats(site_stats, role, sites, weights):
    store = site_stats[role]
    for site, weight in zip(sites, weights):
        site = int(site)
        weight = float(weight)
        if site not in store:
            store[site] = {"sum": 0.0, "n": 0, "max": -np.inf}
        store[site]["sum"] += weight
        store[site]["n"] += 1
        if weight > store[site]["max"]:
            store[site]["max"] = weight


def update_special_stats(special_stats, role, special):
    for token_name, weight in special.items():
        key = (role, token_name)
        if key not in special_stats:
            special_stats[key] = {"sum": 0.0, "n": 0, "max": -np.inf}
        special_stats[key]["sum"] += float(weight)
        special_stats[key]["n"] += 1
        if float(weight) > special_stats[key]["max"]:
            special_stats[key]["max"] = float(weight)


def extract_residue_attention_summary():
    run_config = json.loads((RESULT_DIR / "run_config.json").read_text())
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

    model = SerumGateModel(ckpt["model_config"])
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    task_cols = run_config["task_cols"]
    frames = load_fixed_split_frames(
        data_dir=Path(run_config["data_dir"]),
        strict_group_resplit_enabled=False,
        preserve_test_split=True,
        refit_train_valid=True,
        allow_task_overlap=True,
        task_cols=task_cols,
        seed=run_config.get("seed", 42),
    )
    train_frame = frames["train"].copy()
    if MAX_ROWS_FOR_DEBUG is not None:
        train_frame = train_frame.iloc[:MAX_ROWS_FOR_DEBUG].copy()
    print(f"[info] train rows for attention extraction = {len(train_frame)}")

    vocabs = SerumGateVocabs(
        passage_to_id=ckpt["passage_to_id"],
        subtype_to_id=ckpt["subtype_to_id"],
    )
    embedding_files = required_embedding_files({"train": train_frame}, include_na_embeddings=False)
    embedding_dir = validate_embedding_files(Path(run_config["embedding_dir"]), embedding_files)

    embeddings = {}
    for filename in tqdm(embedding_files, desc="load embeddings", unit="file", dynamic_ncols=True, file=sys.stdout):
        value = torch.load(embedding_dir / filename, map_location="cpu", weights_only=False)
        embeddings[filename.removesuffix(".pt")] = torch.as_tensor(value).float()

    loader = build_loader(
        frame=train_frame,
        vocabs=vocabs,
        embeddings=embeddings,
        batch_size=1,
        shuffle=False,
        max_queries_per_task=run_config["max_queries_per_task"],
        task_cols=task_cols,
        align_ha_embeddings=False,
        include_na_embeddings=False,
    )

    site_stats = {"serum": {}, "virus": {}}
    special_stats = {}
    total_rows = len(train_frame)
    seen_rows = 0
    start = time.time()
    pbar = tqdm(loader, total=len(loader), desc="residue attention", unit="chunk", dynamic_ncols=True, file=sys.stdout)

    with torch.no_grad():
        for chunk_i, (batch, items) in enumerate(pbar, start=1):
            item = items[0]
            meta = item["query_meta"].reset_index(drop=True)
            batch = move_batch(batch, device)

            # Run full model inference first so the extraction matches inference-time modules/state.
            _ = model(batch)

            ref_w = lowrank_attention_weights(model, batch.reference_ha, batch.reference_ha_mask)[0].detach().cpu().numpy()
            ref_len = int(batch.reference_ha_mask[0].detach().cpu().sum().item())
            ref_w = ref_w[:ref_len]
            ref_sites, _, ref_residue_w, ref_special = split_attention_to_aligned_residue_sites(meta.iloc[0]["serumHA"], ref_w)
            update_site_stats(site_stats, "serum", ref_sites, ref_residue_w)
            update_special_stats(special_stats, "serum", ref_special)

            q_matrix = batch.query_ha.reshape(-1, batch.query_ha.shape[-2], batch.query_ha.shape[-1])
            q_mask = batch.query_ha_mask.reshape(-1, batch.query_ha_mask.shape[-1])
            q_w_all = lowrank_attention_weights(model, q_matrix, q_mask).detach().cpu().numpy()
            q_mask_cpu = q_mask.detach().cpu().numpy()

            for row_i in range(len(meta)):
                q_len = int(q_mask_cpu[row_i].sum())
                q_w = q_w_all[row_i, :q_len]
                q_sites, _, q_residue_w, q_special = split_attention_to_aligned_residue_sites(meta.iloc[row_i]["virusHA"], q_w)
                update_site_stats(site_stats, "virus", q_sites, q_residue_w)
                update_special_stats(special_stats, "virus", q_special)

            seen_rows += len(meta)
            elapsed = time.time() - start
            pbar.set_postfix_str(f"rows={seen_rows}/{total_rows}, {seen_rows / max(elapsed, 1e-6):.1f} rows/s")
            if chunk_i % PRINT_EVERY == 0 or chunk_i == len(loader):
                print(f"[progress] chunk {chunk_i}/{len(loader)}, rows {seen_rows}/{total_rows}", flush=True)

    residue_rows = []
    for role, values in site_stats.items():
        for site, stat in values.items():
            residue_rows.append({
                "role": role,
                "site": int(site),
                "mean_attention": stat["sum"] / stat["n"],
                "max_attention": stat["max"],
                "n": stat["n"],
            })
    residue_df = pd.DataFrame(residue_rows).sort_values(["role", "site"]).reset_index(drop=True)
    residue_df.to_csv(RESIDUE_ATTENTION_PATH, index=False)

    special_rows = []
    for (role, token_name), stat in special_stats.items():
        special_rows.append({
            "role": role,
            "token": token_name,
            "mean_attention": stat["sum"] / stat["n"],
            "max_attention": stat["max"],
            "n": stat["n"],
        })
    special_df = pd.DataFrame(special_rows).sort_values(["role", "token"]).reset_index(drop=True)
    special_df.to_csv(SPECIAL_ATTENTION_PATH, index=False)

    ok, reason = is_valid_residue_attention_table(RESIDUE_ATTENTION_PATH)
    if not ok:
        raise RuntimeError(f"Wrote invalid residue attention table: {reason}")
    print(f"[done] wrote {RESIDUE_ATTENTION_PATH}")
    print(f"[done] wrote {SPECIAL_ATTENTION_PATH}")
    print(residue_df.groupby("role")["site"].nunique())
    return residue_df, special_df

ok, reason = is_valid_residue_attention_table(RESIDUE_ATTENTION_PATH)
print(f"[check] {RESIDUE_ATTENTION_PATH.name}: {reason}")
if FORCE_RECOMPUTE or not ok:
    residue_attention_df, special_attention_df = extract_residue_attention_summary()
else:
    residue_attention_df = pd.read_csv(RESIDUE_ATTENTION_PATH)
    special_attention_df = pd.read_csv(SPECIAL_ATTENTION_PATH) if SPECIAL_ATTENTION_PATH.is_file() else pd.DataFrame()
    print(f"[info] using existing valid residue attention table: {RESIDUE_ATTENTION_PATH}")


In [ ]:
def load_and_validate_residue_attention_table():
    ok, reason = is_valid_residue_attention_table(RESIDUE_ATTENTION_PATH)
    if not ok:
        raise ValueError(f"Residue attention table is invalid: {reason}")
    df = pd.read_csv(RESIDUE_ATTENTION_PATH)
    df["site"] = pd.to_numeric(df["site"], errors="raise").astype(int)
    df["mean_attention"] = pd.to_numeric(df["mean_attention"], errors="raise")
    df["max_attention"] = pd.to_numeric(df["max_attention"], errors="raise")
    df["n"] = pd.to_numeric(df["n"], errors="raise").astype(int)
    return df

residue_attention_df = load_and_validate_residue_attention_table()
special_attention_df = pd.read_csv(SPECIAL_ATTENTION_PATH) if SPECIAL_ATTENTION_PATH.is_file() else pd.DataFrame()

role_colors = {"serum": "#3B6EA8", "virus": "#D46A2C"}
role_labels = {"serum": "Serum HA", "virus": "Virus HA"}

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for role in ["serum", "virus"]:
    df = residue_attention_df[residue_attention_df["role"] == role].sort_values("site")
    axes[0].plot(
        df["site"],
        df["mean_attention"],
        label=role_labels[role],
        color=role_colors[role],
        linewidth=1.1,
        marker=".",
        markersize=2.0,
        alpha=0.9,
    )
axes[0].set_ylabel("Mean residue attention")
axes[0].set_title("Raw residue-site mean attention, CLS/EOS removed, no smoothing")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.25)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

for role in ["serum", "virus"]:
    df = residue_attention_df[residue_attention_df["role"] == role].sort_values("site")
    axes[1].plot(
        df["site"],
        df["max_attention"],
        label=role_labels[role],
        color=role_colors[role],
        linewidth=1.1,
        marker=".",
        markersize=2.0,
        alpha=0.9,
    )
    top_sites = df.nlargest(10, "max_attention")
    axes[1].scatter(top_sites["site"], top_sites["max_attention"], color="#B22222", s=20, zorder=3)
    for _, row in top_sites.iterrows():
        axes[1].text(row["site"], row["max_attention"], str(int(row["site"])), fontsize=7, ha="center", va="bottom", rotation=45)
axes[1].set_xlabel("Aligned HA residue site")
axes[1].set_ylabel("True max residue attention")
axes[1].set_title("Raw residue-site true max attention across train+valid, no top-k approximation")
axes[1].legend(frameon=False)
axes[1].grid(axis="y", alpha=0.25)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()
line_png = OUT_DIR / "train_site_residue_mean_and_true_max_attention_line.png"
line_svg = OUT_DIR / "train_site_residue_mean_and_true_max_attention_line.svg"
plt.savefig(line_png, dpi=300, bbox_inches="tight", transparent=True)
plt.savefig(line_svg, bbox_inches="tight", transparent=True)
plt.show()

top50 = residue_attention_df.sort_values(["role", "max_attention"], ascending=[True, False]).groupby("role", as_index=False).head(50)
top50.to_csv(OUT_DIR / "top50_residue_true_max_attention_sites.csv", index=False)

print("saved:")
print(line_png)
print(line_svg)
print(OUT_DIR / "top50_residue_true_max_attention_sites.csv")

if not special_attention_df.empty:
    print("\nSpecial-token attention summary, not mapped to residue sites:")
    print(special_attention_df.to_string(index=False))

print("\nTop residue true max serum sites:")
print(residue_attention_df[residue_attention_df["role"] == "serum"].nlargest(15, "max_attention").to_string(index=False))
print("\nTop residue true max virus sites:")
print(residue_attention_df[residue_attention_df["role"] == "virus"].nlargest(15, "max_attention").to_string(index=False))
